##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [2]:
!pip install transformers torch torchvision pillow scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 728.2 kB/s eta 0:00:14
   - -------------------------------------- 0.5/10.6 MB 728.2 kB/s eta 0:00:14
   -- ------------------------------------- 0.8/10.6 MB 699.0 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/10.6 

In [3]:
from transformers import AutoImageProcessor, AutoModel
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
from PIL import Image
import torch
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

# Load DINOv2 model
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
model = AutoModel.from_pretrained("facebook/dinov2-small")

# Download Oxford-IIIT Pet dataset
dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True
)

# Select 20 images
images = []
true_labels = []

cat_count = 0
dog_count = 0

for img, label in dataset:

    # In this dataset:
    # labels 0-11 تقريباً cats
    # labels 12-36 dogs

    if label < 12 and cat_count < 10:
        images.append(img)
        true_labels.append(0)  # cat
        cat_count += 1

    elif label >= 12 and dog_count < 10:
        images.append(img)
        true_labels.append(1)  # dog
        dog_count += 1

    if len(images) == 20:
        break

print(f"Total images selected: {len(images)}")

# Extract CLS tokens
features = []

for image in images:

    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    cls_token = outputs.last_hidden_state[:, 0, :]
    features.append(cls_token.squeeze().numpy())

# Convert to numpy array
features = np.array(features)

# Apply KMeans clustering
kmeans = KMeans(n_clusters=2, random_state=42)

pred_clusters = kmeans.fit_predict(features)

print("\nPredicted Clusters:")
print(pred_clusters)

print("\nTrue Labels:")
print(true_labels)

# Clustering accuracy
acc1 = accuracy_score(true_labels, pred_clusters)
acc2 = accuracy_score(true_labels, 1 - pred_clusters)

best_acc = max(acc1, acc2)

print(f"\nBest Clustering Accuracy: {best_acc * 100:.2f}%")

c:\Users\2W1\anaconda3\envs\depth-anything\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\2W1\anaconda3\envs\depth-anything\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\2W1\.cache\huggingface\hub\models--facebook--dinov2-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to a

Total images selected: 20

Predicted Clusters:
[0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]

True Labels:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Best Clustering Accuracy: 100.00%


### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch

# Load model and processor
processor = AutoImageProcessor.from_pretrained(
    "facebook/dinov2-small-imagenet1k-1-layer"
)

model = AutoModelForImageClassification.from_pretrained(
    "facebook/dinov2-small-imagenet1k-1-layer"
)

# Load image
image = Image.open("data/grand_piano.jpg").convert("RGB")

# Preprocess image
inputs = processor(images=image, return_tensors="pt")

# Predict
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

# Get predicted class
predicted_class_idx = logits.argmax(-1).item()

predicted_label = model.config.id2label[predicted_class_idx]

print("Predicted Class:")
print(predicted_label)

c:\Users\2W1\anaconda3\envs\depth-anything\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\2W1\.cache\huggingface\hub\models--facebook--dinov2-small-imagenet1k-1-layer. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 225/225 [00:00<00:00, 3760.48it/s]


Predicted Class:
grand piano, grand
